# 강의 04 · 실습 3 — 서브그래프 모듈화 · (6) 고난도 III

## 1. 문제상황

- 온라인 서점이 해외 배송을 시작하면서 고객센터에 영어 메일이 섞여 들어옵니다.
- 한국어 메일은 국내 분류 팀의 분류 그래프로, 영어 메일은 해외 분류 팀의 분류 그래프로 보내야 합니다.
- 국내 팀의 그래프는 고객센터와 같은 상태 키(email·category)를 쓰지만, 해외 팀의 그래프는 다른 키(text·label)를 쓰고 분류 결과도 영어 단어(shipping·damage·refund)로 돌려줍니다.
- 고객센터 그래프는 어느 팀이 분류했든 같은 한국어 분류(배송·파손·환불)로 담당자를 배정해야 합니다.
- 지금은 담당자가 메일의 언어를 눈으로 보고 두 프로그램 중 하나를 골라 돌립니다.

## 2. 문제와 목표

- **문제**: 언어에 따라 서로 다른 팀의 분류 그래프를 사람이 골라 돌리고, 두 그래프의 상태 키와 분류 결과의 언어가 달라 결과를 한 곳에 모을 수 없습니다.
- **목표**
  - 고객센터 부모 그래프가 메일의 언어를 스스로 판단해 국내 팀과 해외 팀의 분류 그래프 중 하나로 보냅니다.
    - 언어 판정의 기준: 본문에 한글이 있으면 한국어, 없으면 영어
  - 어느 팀이 분류했든 한국어 분류(배송·파손·환불)로 담당자를 배정하게 만듭니다.
    - 배정 메시지는 부모 상태의 `handled` 키에 씁니다.
    - 해외 팀 분류 결과의 대응: shipping → 배송, damage → 파손, refund → 환불, 그 밖은 「기타」
  - 두 팀의 분류 그래프는 고치지 않고 자식 그래프로 얹습니다.
  - 메일 두 통의 문면은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**:
  - 한국어 파손 메일과 영어 배송 메일을 넣었을 때, 두 메일이 서로 다른 팀의 자식을 거치고,
  - 두 메일 모두 마지막에 한국어 분류로 된 배정 메시지가 만들어지는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항

(요구사항을 번호 목록으로 적습니다.)

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 주어진 것

라이브러리를 불러오고 모델을 준비합니다. 아래 코드 셀은 채워져 있으므로 그대로 실행합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")


# 주어진 자료
EMAILS = [
    "주문한 책이 찢어진 채로 왔습니다. 교환해 주세요.",
    "My order has not arrived yet. Could you check the shipping status?",
]
EN_TO_KO = {"shipping": "배송", "damage": "파손", "refund": "환불"}


In [ ]:
# 여기에 「5. 코드 골격」의 표 순서대로 코드를 작성합니다. 단계마다 셀을 나눕니다.

## 7. 실행 결과 확인

스스로 작성한 코드의 실행 결과에서 다음 세 가지를 확인합니다.

1. 두 메일이 서로 다른 자식을 거칩니다. 언어를 판단하는 노드와 자식을 고르는 조건부 엣지가 실행 결과에서 구별됩니다.
2. 영어 메일에서는 해외 팀의 자식이 영어 분류 결과를 돌려주고, 부모의 상태에는 한국어 분류가 들어갑니다.
3. 두 메일의 최종 상태 모두 handled가 한국어 분류로 채워집니다. 어느 팀의 자식이 분류했는지는 배정 단계가 모릅니다.